# Lección 13.5 — Cómo se entrena un LLM: pre-entrenamiento, ajuste por instrucciones, RLHF/RLAIF

> **Módulo 13 · Modelos generativos y LLMs** · Nivel 5 — Héroe
> ⏱️ Tiempo estimado: 90–110 min (lección con ejercicios intercalados ≈ 65 min · ejercicios finales ≈ 30 min)
> 📋 Requisitos previos: Lección 11.2 (tokens y embeddings), Lección 12.4 (GPT es un decoder), Lección 12.5 (leyes de escalamiento), Lección 13.1 (modelos generativos)

### 🎯 Objetivos
Al terminar esta lección podrás:
- Describir las **tres etapas** con las que se fabrica un asistente tipo ChatGPT: pre-entrenamiento, ajuste por instrucciones y RLHF/RLAIF.
- Explicar por qué **predecir el siguiente token** basta para que un modelo aprenda gramática y conocimiento.
- Leer una **pérdida** y una **perplejidad** y entrenar un mini-modelo de lenguaje en tu computadora.
- Predecir el efecto de la tasa de aprendizaje, el tamaño de lote y la longitud de contexto en el pre-entrenamiento.
- Reconocer el riesgo de **reward hacking** cuando se optimiza un modelo de recompensa.

### 🧭 Cómo usar este notebook
| Icono | Significa |
|---|---|
| 📖 | Lee: explicación del concepto |
| ▶️ | Ejecuta la celda y observa el resultado |
| 🔮 | **Predice** antes de ejecutar: escribe tu apuesta y luego compruébala |
| ✋ | **Tu turno**: ejercicio. Las respuestas están ocultas en "👀 Ver respuesta" |
| ✏️ | Modifica un valor en el código y vuelve a ejecutar |

> 💡 No necesitas saber programar. El código está para que **veas** los conceptos en acción; solo tendrás que cambiar números.

## ▶️ Preparación
Ejecuta esta celda una vez (`Shift + Enter`). Carga PyTorch y GPT-2 small (124 millones de parámetros; la primera vez se descarga, ~500 MB).

In [ ]:
import math
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM, AutoTokenizer

torch.manual_seed(0)                      # semilla fija: así tus números coinciden con las respuestas
device = "cuda" if torch.cuda.is_available() else "cpu"
tok = AutoTokenizer.from_pretrained("gpt2")
gpt2 = AutoModelForCausalLM.from_pretrained("gpt2").eval()
plt.rcParams["figure.figsize"] = (8, 4)
print("✅ Todo listo. GPT-2 tiene", f"{sum(p.numel() for p in gpt2.parameters()):,}", "parámetros")

---
## 1. 📖 La intuición: cómo se forma un médico

Piensa en cómo se forma un buen médico:

```
 1. UNIVERSIDAD             2. RESIDENCIA                 3. EVALUACIÓN DE PACIENTES
 Lee miles de libros   ──►  Atiende casos reales     ──►  Pacientes y supervisores dicen
 (sabe MUCHO, pero no       con un tutor que le dice      qué consultas fueron mejores;
  sabe atender a nadie)     "así se responde"             ajusta su estilo con esa opinión
```

Un **LLM** (*Large Language Model*, modelo de lenguaje grande) se fabrica igual:

| Etapa | En el médico | En el LLM |
|---|---|---|
| **1. Pre-entrenamiento** (*pre-training*) | Leer la biblioteca entera | Predecir el siguiente token en billones de tokens de internet, libros y código |
| **2. Ajuste por instrucciones** (*instruction tuning* / SFT) | Residencia con tutor | Aprender de miles de ejemplos "instrucción → buena respuesta" escritos por personas |
| **3. RLHF / RLAIF** | Opinión de pacientes | Aprender de **preferencias** ("la respuesta A es mejor que la B") dadas por humanos (HF) o por otra IA (AIF) |

### ✋ Ejercicio 1 — ¿Qué etapa es?
Clasifica cada situación en la etapa 1, 2 o 3:
- a) Una empresa paga a 40 personas para que, ante la misma pregunta, elijan cuál de dos respuestas es más útil.
- b) Se descargan 15 billones de tokens de páginas web y se le pide al modelo adivinar cada palabra siguiente.
- c) Se escriben 20,000 ejemplos como *"Instrucción: resume este correo. Respuesta: …"*.
- d) Otro modelo, guiado por una lista de principios ("sé honesto, no ayudes a dañar a nadie"), juzga cuál respuesta es mejor.

✍️ *Tu respuesta:* a → _ · b → _ · c → _ · d → _

<details><summary><b>👀 Ver respuesta</b></summary>

a → **3** (RLHF: preferencias humanas) · b → **1** (pre-entrenamiento) · c → **2** (ajuste por instrucciones) · d → **3** también, pero en su versión **RLAIF**: la retroalimentación la da una IA siguiendo una "constitución" de principios. Es más barata que pagar a personas, aunque hereda los sesgos del modelo juez.
</details>

---
## 2. 📖 El concepto: tres etapas muy distintas en datos y costo

| | 1. Pre-entrenamiento | 2. Ajuste por instrucciones | 3. RLHF / RLAIF |
|---|---|---|---|
| **Objetivo** | Predecir el siguiente token | Imitar respuestas de ejemplo | Preferir respuestas mejor valoradas |
| **Datos** | Texto crudo, sin etiquetas | Pares instrucción → respuesta | Comparaciones "A mejor que B" |
| **Cantidad** | Billones de tokens (≈10¹³) | Miles a cientos de miles de ejemplos | Decenas a cientos de miles de comparaciones |
| **Costo** | Enorme: miles de GPUs durante semanas o meses | Pequeño en comparación | Mediano |
| **Lo que aporta** | Gramática, conocimiento, razonamiento | El **formato** de asistente: seguir instrucciones | El **estilo**: útil, honesto, seguro |

> 💡 Casi todo lo que el modelo **sabe** se aprende en la etapa 1. Las etapas 2 y 3 le enseñan **cómo comportarse**. Por eso a un modelo recién salido del pre-entrenamiento se le llama **modelo base** (*base model*), y al final, **modelo de chat** o **asistente**.

GPT-2 (2019) es un **modelo base**: solo pasó por la etapa 1. ¿Qué crees que hace si le hacemos una pregunta?

### 🔮 Ejercicio 2 — Un modelo base frente a una pregunta
Le daremos a GPT-2 el texto `"Question: What is the capital of France?\nAnswer:"` y le pediremos que continúe, tres veces con azar distinto.

¿Responderá siempre "Paris"? ¿Qué más podría escribir?

✍️ *Mi predicción:*

In [ ]:
def continuar(prompt, semilla=0, n_tokens=30):
    # GPT-2 continúa el texto eligiendo tokens al azar según sus probabilidades (lo verás a fondo en la Lección 13.6)
    torch.manual_seed(semilla)
    ids = tok(prompt, return_tensors="pt")
    out = gpt2.generate(**ids, max_new_tokens=n_tokens, do_sample=True, top_p=0.9,
                        pad_token_id=tok.eos_token_id)
    return tok.decode(out[0][ids.input_ids.shape[1]:])

for s in range(3):
    print(f"--- intento {s}:", repr(continuar("Question: What is the capital of France?\nAnswer:", s)))

<details><summary><b>👀 Ver respuesta</b></summary>

**No responde como un asistente.** En los tres intentos GPT-2 escribe frases que *suenan* a respuesta ("The capital of France is…"), pero divaga: en un intento dice que la capital de Francia es "the Marseille Opera House", en otro mezcla Francia con Gran Bretaña, y ninguno dice "Paris".

Un modelo base solo **continúa el texto como lo haría un documento de internet**. No tiene la costumbre de "responder y detenerse": eso se aprende en las etapas 2 y 3. (Además, GPT-2 small es pequeño y su conocimiento es frágil; lo verás en la Lección 13.9.)
</details>

---
## 3. 📖 Cómo funciona por dentro el pre-entrenamiento

La tarea es siempre la misma: **dado un texto, adivinar el siguiente token** (Lección 11.2). No hace falta etiquetar nada: el propio texto trae la respuesta.

```
 Texto:        "el perro come el pan"
 Entrada  ──►  "el perro come el"     Respuesta correcta: " pan"
 Entrada  ──►  "el perro come"        Respuesta correcta: " el"
 ...un ejemplo de entrenamiento por cada posición del texto
```

El modelo produce una **probabilidad** para cada token posible. La **pérdida** (*loss*, Lección 3.2) es la **entropía cruzada**: la "sorpresa" que se lleva el modelo con la respuesta correcta.

$$\text{pérdida} = -\log(\text{probabilidad que el modelo le dio a la respuesta correcta})$$

- Si le dio probabilidad 1 → pérdida 0 (nada sorprendido).
- Si le dio probabilidad 0.01 → pérdida 4.6 (muy sorprendido).

Calculémoslo a mano con numpy:

In [ ]:
# El modelo debe adivinar la letra que sigue a "el perr". La respuesta correcta es "o".
probabilidades = {"o": 0.70, "a": 0.20, "e": 0.05, "x": 0.05}   # lo que dice el modelo
correcta = "o"

perdida = -np.log(probabilidades[correcta])
print(f"Probabilidad que dio a la correcta: {probabilidades[correcta]}  →  pérdida = {perdida:.3f}")
print(f"Perplejidad = e^pérdida = {np.exp(perdida):.2f}  (≈ 'entre cuántas opciones duda')")

La **perplejidad** (*perplexity*) es $e^{\text{pérdida}}$: se interpreta como **entre cuántas opciones igual de probables está dudando** el modelo. Perplejidad 1 = certeza total.

### 🔮 Ejercicio 3 — El modelo que no sabe nada
Antes de entrenar, un modelo que reparte la probabilidad **por igual** entre 30 caracteres posibles, ¿qué pérdida y qué perplejidad tiene?

✍️ *Mi predicción:* pérdida ≈ ___ · perplejidad ≈ ___

In [ ]:
V = 30   # ✏️ prueba con 50,257 (el vocabulario de GPT-2)
p_uniforme = 1 / V
print(f"pérdida = {-np.log(p_uniforme):.3f} · perplejidad = {np.exp(-np.log(p_uniforme)):.0f}")

<details><summary><b>👀 Ver respuesta</b></summary>

Pérdida = ln(30) ≈ **3.40** y perplejidad = **30**: duda entre las 30 opciones por igual. Con el vocabulario de GPT-2 (50,257 tokens) la pérdida inicial sería ≈ **10.8**. Recuerda estos números: **una pérdida al inicio del entrenamiento cercana a ln(tamaño del vocabulario) es señal de que todo está bien conectado.**
</details>

### ▶️ Un mini-LLM en tu computadora
Vamos a pre-entrenar un modelo de lenguaje **de caracteres** diminuto (≈20,000 parámetros, contra los 124 millones de GPT-2). Primero, el "internet" de nuestro modelo: 1,500 frases generadas con una gramática sencilla.

In [ ]:
rng = np.random.default_rng(0)
sujetos = ["el perro", "el gato", "la niña", "el niño", "la abuela", "el abuelo", "mi hermana", "el vecino", "la maestra", "el panadero"]
verbos  = ["come", "mira", "busca", "dibuja", "cuida", "encuentra", "lleva", "quiere"]
objetos = ["el pan", "la luna", "una flor", "el libro", "la pelota", "un regalo", "la casa", "el camino"]
lugares = ["en el campo", "por la mañana", "en la cocina", "junto al río", "bajo la lluvia", "con calma", "desde la ventana", ""]

frases = [f"{rng.choice(sujetos)} {rng.choice(verbos)} {rng.choice(objetos)} {rng.choice(lugares)}".strip() + "."
          for _ in range(1500)]
corpus = " ".join(frases)

caracteres = sorted(set(corpus))                      # nuestro "vocabulario": cada carácter es un token
a_id = {c: i for i, c in enumerate(caracteres)}
datos = torch.tensor([a_id[c] for c in corpus])
print(f"Corpus: {len(corpus):,} caracteres · vocabulario: {len(caracteres)} tokens")
print(corpus[:160], "...")

El modelo mira los **K caracteres anteriores** (su *ventana de contexto*) y predice el siguiente. Separamos el 10 % final del corpus como **validación**, para ver si aprende de verdad o solo memoriza (Lección 4.4).

La función `entrenar` hace todo: arma el modelo, lo entrena y, en algunos pasos, imprime la pérdida y una muestra de texto generado. No necesitas entender cada línea.

In [ ]:
def entrenar(K=8, lr=3e-3, lote=64, pasos=1500, mostrar=(0, 100, 300, 1000, 1500)):
    torch.manual_seed(0)
    X = torch.stack([datos[i:i + K] for i in range(len(datos) - K)])   # ventanas de K caracteres
    Y = datos[K:]                                                       # el carácter que sigue a cada ventana
    n = int(0.9 * len(X))
    modelo = nn.Sequential(nn.Embedding(len(caracteres), 16), nn.Flatten(),
                           nn.Linear(16 * K, 128), nn.Tanh(), nn.Linear(128, len(caracteres)))
    opt = torch.optim.Adam(modelo.parameters(), lr=lr)

    def generar(inicio="el perro", n_car=90):
        g = torch.Generator().manual_seed(1)
        ctx = [a_id[" "]] * K + [a_id[c] for c in inicio]
        texto = inicio
        with torch.no_grad():
            for _ in range(n_car):
                p = torch.softmax(modelo(torch.tensor([ctx[-K:]])), -1)[0]
                i = torch.multinomial(p, 1, generator=g).item()
                ctx.append(i); texto += caracteres[i]
        return texto

    for paso in range(pasos + 1):
        idx = torch.randint(0, n, (lote,))                                  # un lote al azar
        perdida = nn.functional.cross_entropy(modelo(X[idx]), Y[idx])
        opt.zero_grad(); perdida.backward(); opt.step()
        if paso in mostrar:
            with torch.no_grad():
                val = nn.functional.cross_entropy(modelo(X[n:]), Y[n:]).item()
            print(f"paso {paso:>5} · pérdida train {perdida.item():.2f} · validación {val:.2f} │ {generar()!r}")
    with torch.no_grad():
        val = nn.functional.cross_entropy(modelo(X[n:]), Y[n:]).item()   # pérdida de validación final
    modelo.generar = generar
    return modelo, val

### 🔮 Ejercicio 4 — Mira aprender a tu modelo
Antes de ejecutar:
1. ¿Qué pérdida esperas en el **paso 0**? (Pista: Ejercicio 3 y el tamaño del vocabulario que se imprimió arriba).
2. ¿Qué aprenderá primero: palabras completas, o espacios y letras frecuentes?

✍️ *Mis predicciones:*

In [ ]:
mini, _ = entrenar()

<details><summary><b>👀 Ver respuesta</b></summary>

1. En el paso 0 la pérdida es **≈3.2**, muy cerca de ln(25) ≈ 3.22: el vocabulario tiene **25** caracteres y el modelo sin entrenar reparte la probabilidad casi por igual. El texto es una sopa de letras.
2. Aprende **primero lo frecuente**: en el paso 100 (pérdida ≈0.9) ya alterna letras y espacios y aparecen trozos como "en el", "la". Hacia el paso 300 (≈0.3–0.4) salen palabras completas ("el campo", "el abuelo mira bajo la lluvia"), y al final escribe frases casi correctas: *"el perro lleva el libro. el panadero cuida una flor junto al río."*

La pérdida de validación (≈0.26) queda pegada a la de entrenamiento: no memorizó, **aprendió la gramática** de su pequeño mundo. Nadie le explicó qué es un sujeto o un verbo: lo dedujo de predecir caracteres.
</details>

### 📖 Lo mismo, a escala real: la perplejidad de GPT-2
GPT-2 se entrenó exactamente así, pero con 40 GB de texto en inglés y una red Transformer (Lección 12.4). Podemos pedirle su **pérdida** sobre cualquier texto: es una medida de qué tan "normal" le parece.

### 🔮 Ejercicio 5 — ¿Qué texto sorprende más a GPT-2?
Ordena de **menor a mayor** perplejidad (de "le parece más normal" a "le parece más raro"):
- A: una frase normal en inglés
- B: la misma frase con las palabras desordenadas
- C: la misma idea en **español**
- D: una frase gramatical pero sin sentido ("Colorless green ideas…")

✍️ *Mi orden:*

In [ ]:
def perplejidad(texto):
    ids = tok(texto, return_tensors="pt").input_ids
    with torch.no_grad():
        perdida = gpt2(ids, labels=ids).loss.item()      # entropía cruzada media por token
    return perdida, math.exp(perdida), ids.shape[1]

textos = {
    "A normal ": "The cat sat on the mat and fell asleep in the warm afternoon sun.",
    "B revuelta": "sun afternoon the cat warm mat in sat the fell on asleep and.",
    "C español": "El gato se sentó en la alfombra y se durmió bajo el sol de la tarde.",
    "D sin sentido": "Colorless green ideas sleep furiously while the purple economy dances.",
}
for nombre, t in textos.items():
    l, ppl, n = perplejidad(t)
    print(f"{nombre:<14} pérdida {l:5.2f} · perplejidad {ppl:8.1f} · {n} tokens")

<details><summary><b>👀 Ver respuesta</b></summary>

**A (≈40) < C (≈193) < D (≈2,040) < B (≈3,070).**
- La frase normal es la menos sorprendente: la vio "miles de veces" en su entrenamiento.
- El español le cuesta más porque GPT-2 vio poquísimo español. Fíjate además en que la misma idea ocupa **26 tokens** en español contra **15** en inglés: su vocabulario se aprendió con texto inglés, así que trocea el español en pedazos más pequeños. Más tokens = más costo y más contexto ocupado (lo retomamos en la Lección 13.9).
- La frase sin sentido es gramatical, pero las combinaciones de palabras son rarísimas; la revuelta rompe hasta la gramática.

👉 **La perplejidad mide lo que el modelo aprendió de sus datos**, no si un texto es verdadero o útil.
</details>

---
## 4. 📖 Hiperparámetros del pre-entrenamiento

| Hiperparámetro | Qué controla | Si lo **subes** ⬆️ | Si lo **bajas** ⬇️ | Punto de partida |
|---|---|---|---|---|
| Tasa de aprendizaje (`lr`) | Tamaño de cada paso del optimizador | Aprende rápido, pero puede **diverger** (la pérdida salta o sube) | Estable pero **lentísimo** | 1e-3–3e-3 (modelos chicos) · 1e-4–6e-4 (LLMs, con *warmup*) |
| Tamaño de lote (`lote`, *batch size*) | Ejemplos por paso | Gradiente más estable, pasos más caros | Más ruido, pasos baratos | 32–256 (LLMs: millones de tokens por lote) |
| Longitud de contexto (`K`, *context length*) | Cuántos tokens anteriores ve | Captura dependencias largas, más cómputo | Solo patrones locales | 8–64 aquí · 1,024 en GPT-2 · 100k+ en LLMs actuales |
| Pasos / tokens vistos | Cuánto entrena | Mejor, hasta que se agotan los datos (y luego memoriza) | Subentrenado | Regla de Chinchilla: ≈20 tokens por parámetro (Lección 12.5) |
| Tamaño del modelo | Capacidad | Más capacidad, si hay datos suficientes | Se queda corto | Según presupuesto (leyes de escalamiento, Lección 12.5) |

### 🔮 Ejercicio 6 — Un contexto de solo 2 caracteres
Si el modelo solo ve los **2** caracteres anteriores (`K=2`), ¿qué pasará con la pérdida final y con el texto generado?

✍️ *Mi predicción:*

In [ ]:
K_CONTEXTO = 2   # ✏️ prueba también 4 y 16
_ = entrenar(K=K_CONTEXTO, mostrar=(1500,))

<details><summary><b>👀 Ver respuesta</b></summary>

Con `K=2` la pérdida final se queda en **≈0.9** (contra ≈0.26 con `K=8`) y el texto es un revoltijo de sílabas plausibles: *"el pan junto lunto cami herma…"*. Con solo 2 letras de contexto no puede saber en qué palabra está, así que solo aprende qué letras suelen ir juntas.

✏️ Si pruebas `K=4` llega a ≈0.39 y con `K=16` a ≈0.27: más contexto ayuda mucho al principio y luego **deja de ayudar**, porque en este corpus las dependencias son cortas (una frase). En textos reales sí hacen falta contextos larguísimos, por eso los LLMs modernos ven decenas o cientos de miles de tokens.
</details>

### 🔮 Ejercicio 7 — La tasa de aprendizaje
Ahora probaremos `lr = 0.1` (30 veces más grande) y `lr = 0.0001` (30 veces más chica) con `K=8`. ¿Cuál de las dos **no aprende** y cuál aprende **demasiado lento**?

✍️ *Mi predicción:*

In [ ]:
for lr in [0.1, 0.0001]:   # ✏️ prueba otros valores
    print(f"=== lr = {lr}")
    _ = entrenar(lr=lr, mostrar=(300, 1500))

<details><summary><b>👀 Ver respuesta</b></summary>

- `lr = 0.1` **no aprende**: la pérdida rebota entre ≈2.4 y ≈2.9, sin bajar de forma sostenida, y el texto sigue siendo basura ("ln porora ln porora…"). Los pasos son tan grandes que el optimizador "salta" por encima del valle (Lección 3.3).
- `lr = 0.0001` aprende **demasiado lento**: baja de forma estable, pero tras 1,500 pasos sigue en ≈1.2 (con `lr=0.003` estaba en ≈0.26).

En LLMs reales esto es crítico: un entrenamiento cuesta millones de dólares, así que se usa un **warmup** (empezar con `lr` pequeña y subirla poco a poco) y luego se va bajando.
</details>

### ✏️ Ejercicio 8 — El tamaño de lote
Cambia `LOTE` a 8 y luego a 512 (dejando 1,500 pasos). ¿Qué cambia en la pérdida de validación? ¿Cuál tarda más por paso?

In [ ]:
LOTE = 8   # ✏️ prueba 8, 64 y 512
_ = entrenar(lote=LOTE, mostrar=(1500,))

<details><summary><b>👀 Ver respuesta</b></summary>

Con el mismo número de pasos:
- `LOTE = 8` → validación ≈**0.37**: cada paso ve pocos ejemplos, el gradiente es ruidoso y el modelo aprende peor.
- `LOTE = 64` → ≈0.26.
- `LOTE = 512` → ≈**0.24**, el mejor, pero cada paso cuesta ~8 veces más cómputo (la celda tarda varias veces más).

Un lote grande da gradientes más estables a cambio de más cómputo por paso. Los LLMs usan lotes de **millones de tokens**, repartidos entre miles de GPUs.
</details>

---
## 5. 📖 Etapa 2: ajuste por instrucciones (SFT)

Después del pre-entrenamiento, el modelo base sabe mucho pero solo **continúa textos**. El **ajuste por instrucciones** (*supervised fine-tuning*, SFT) sigue entrenando el mismo modelo, con la **misma** pérdida de "siguiente token", pero sobre datos con forma de conversación:

```
<|usuario|> Dame tres consejos para dormir mejor.
<|asistente|> 1. Acuéstate y levántate a la misma hora. 2. Evita pantallas una hora antes. 3. Mantén el cuarto fresco y oscuro.<|fin|>
```

- Las marcas `<|usuario|>`, `<|asistente|>`, `<|fin|>` son tokens especiales: la **plantilla de chat** (*chat template*).
- Normalmente la pérdida se calcula **solo sobre la respuesta**: queremos que aprenda a responder, no a imitar preguntas.
- Con unos miles de ejemplos de buena calidad basta para cambiar el **comportamiento**; la calidad importa más que la cantidad.

Veamos a GPT-2 (sin SFT) frente a una instrucción:

In [ ]:
instruccion = "Instruction: Give me three tips to sleep better.\nResponse:"
for s in range(2):
    print(f"--- intento {s}:", repr(continuar(instruccion, s, n_tokens=40)))

### ✋ Ejercicio 9 — Escribe datos de SFT
1. ¿Siguió GPT-2 la instrucción? ¿Por qué?
2. Escribe **dos** ejemplos de entrenamiento (instrucción + respuesta ideal) que le enseñarían a responder bien. ¿Qué características debería tener una "respuesta ideal"?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. **No.** En un intento escribe como si fuera un foro de programación ("set up one of the following in your main script…"); en el otro dice "Sleep well. It's important." y luego cuenta una historia de su infancia. Continúa el texto, no **obedece**: nunca vio el patrón "instrucción → respuesta útil → fin".
2. Por ejemplo:
   - *Instrucción:* "Explica qué es la fotosíntesis a un niño de 8 años." → *Respuesta:* "Las plantas fabrican su comida con la luz del sol, el agua y el aire. Por eso necesitan estar al sol."
   - *Instrucción:* "Traduce al inglés: buenos días." → *Respuesta:* "Good morning."

   Una buena respuesta de SFT: **cumple exactamente lo pedido**, es correcta, tiene el tono y el largo adecuados, y **termina**. Si los ejemplos son largos y divagan, el modelo aprenderá a divagar.
</details>

---
## 6. 📖 Etapa 3: RLHF y RLAIF — aprender de preferencias

Escribir la respuesta perfecta es difícil; **comparar** dos respuestas es fácil. El RLHF (*Reinforcement Learning from Human Feedback*) aprovecha eso:

```
1. El modelo genera varias respuestas a la misma pregunta
2. Personas (RLHF) u otra IA (RLAIF) dicen cuál es mejor     → "B > A"
3. Con esas comparaciones se entrena un MODELO DE RECOMPENSA (reward model):
   una red que da un puntaje a cualquier respuesta
4. Se ajusta el LLM para que produzca respuestas con puntaje alto
   (con PPO, Lección 14.4, o variantes más simples como DPO)
```

Lo veremos con una versión de juguete:
- Nuestro **"modelo de recompensa"** será una función que da puntos por palabras útiles y quita puntos por palabras groseras.
- En lugar de reentrenar GPT-2, usaremos **best-of-n**: generar *n* respuestas y quedarnos con la de mayor recompensa. Es la forma más sencilla de "optimizar" contra un modelo de recompensa.

In [ ]:
BUENAS = {"sorry", "apologize", "refund", "replace", "fix", "help", "please", "money", "back"}
MALAS  = {"annoyed", "no", "not", "don't", "never"}

def recompensa(texto):
    palabras = [p.strip(".,!?()'\"").lower() for p in texto.split()]
    return sum(p in BUENAS for p in palabras) - sum(p in MALAS for p in palabras)

pregunta = "Customer: My order arrived broken. What should I do?\nAgent:"
torch.manual_seed(0)
ids = tok(pregunta, return_tensors="pt")
salidas = gpt2.generate(**ids, max_new_tokens=30, do_sample=True, top_p=0.9,
                        num_return_sequences=6, pad_token_id=tok.eos_token_id)
candidatas = [tok.decode(s[ids.input_ids.shape[1]:]).split("\n")[0].strip() for s in salidas]
for i, c in enumerate(candidatas):
    print(f"[{i}] recompensa {recompensa(c):+d} │ {c}")

### 🔮 Ejercicio 10 — Best-of-n
1. Si eliges **una candidata al azar**, ¿qué recompensa media esperas? ¿Y si eliges siempre **la mejor** de las 6?
2. ¿La candidata ganadora es de verdad la mejor respuesta para el cliente?

✍️ *Mis respuestas:*

In [ ]:
puntajes = [recompensa(c) for c in candidatas]
mejor = int(np.argmax(puntajes))
print(f"Recompensa media al azar: {np.mean(puntajes):+.2f}")
print(f"Best-of-6               : {puntajes[mejor]:+d} → [{mejor}] {candidatas[mejor]}")

<details><summary><b>👀 Ver respuesta</b></summary>

1. Al azar, la recompensa media es **+0.33**; la mejor de 6 obtiene **+2**. Best-of-n **siempre** sube la recompensa: elegir el máximo de varios intentos nunca es peor que elegir uno al azar.
2. **No del todo.** La ganadora ([1]) dice "please" y "money back", pero también *"Please do not ask me to repair my order again"*, que es bastante grosero. La [2] ofrece arreglar el problema, pero empieza "(somewhat annoyed)". Nuestro modelo de recompensa solo cuenta palabras, así que **no entiende** la respuesta. Los modelos de recompensa reales son redes neuronales mucho mejores, pero también imperfectos.
</details>

### 🔮 Ejercicio 11 — Hacer trampa con la recompensa (*reward hacking*)
Un modelo que se optimiza muchísimo contra un modelo de recompensa imperfecto termina encontrando **sus huecos**. Agregamos una séptima candidata, "optimizada" por un modelo tramposo. ¿Qué puntaje crees que obtiene? ¿Ayuda al cliente?

In [ ]:
tramposa = "Sorry sorry, please help! Refund, replace, fix, money back. Sorry, please help."
candidatas_7 = candidatas + [tramposa]
puntajes_7 = [recompensa(c) for c in candidatas_7]
ganadora = int(np.argmax(puntajes_7))
print(f"Ganadora: [{ganadora}] recompensa {puntajes_7[ganadora]:+d} │ {candidatas_7[ganadora]}")

<details><summary><b>👀 Ver respuesta</b></summary>

La tramposa gana con **+12**, muy por encima de cualquier respuesta real (+2 como máximo), y **no ayuda en nada**: es una lista de palabras clave. Eso es *reward hacking*: el modelo encontró la forma de **subir la métrica sin cumplir el objetivo**.

En RLHF real se ve como respuestas aduladoras, demasiado largas o llenas de disculpas. Por eso se penaliza al modelo si se aleja demasiado del original (penalización **KL**) y se renueva el modelo de recompensa. Volverás a ver este problema en el aprendizaje por refuerzo (Lecciones 14.1 y 14.5).
</details>

---
## 7. 📖 Errores comunes y malentendidos

| 🚨 Malentendido / síntoma | Qué pasa en realidad | Qué hacer |
|---|---|---|
| "El modelo base no sirve, no responde preguntas" | Sabe mucho, pero solo continúa textos | Darle el formato adecuado (*prompting*, Lección 13.8) o aplicar SFT |
| La pérdida inicial está muy lejos de ln(vocabulario) | Algo está mal conectado (datos, etiquetas desplazadas) | Revisa que la "respuesta" sea de verdad el **siguiente** token |
| La pérdida salta o se vuelve `nan` | Tasa de aprendizaje demasiado alta | Bajar `lr`, usar *warmup* y recorte de gradiente |
| Pérdida de entrenamiento baja, validación sube | Memorización: pocos datos para el tamaño del modelo | Más datos o modelo más chico (Lección 12.5) |
| "Con SFT le enseño hechos nuevos" | El SFT cambia sobre todo el **comportamiento**; meter conocimiento nuevo así es poco fiable y puede causar alucinaciones | Para conocimiento, mejor RAG (Lección 13.8) |
| El modelo se vuelve adulador o repite frases "que gustan" | *Reward hacking*: optimizó de más contra la recompensa | Penalizar alejarse del modelo original (KL), mejores datos de preferencia |
| Evaluar solo con la pérdida/perplejidad | Baja perplejidad ≠ respuestas útiles o verdaderas | Evaluar con tareas y personas reales |

### 🔮 Ejercicio 12 — Diagnóstico rápido
Entrenas tu propio mini-LLM con un vocabulario de 50 caracteres y en el **paso 0** la pérdida es **0.02**. ¿Es una gran noticia?

✍️ *Mi respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

**No, es una señal de alerta.** Un modelo sin entrenar debería tener una pérdida cercana a ln(50) ≈ **3.9**. Una pérdida casi cero desde el inicio significa que el modelo **ve la respuesta**: típicamente un error al armar los datos (la "respuesta" está dentro de la entrada). Es la versión LLM de la **fuga de datos** (Lección 2.6).
</details>

---
## 8. 📋 Ficha del proceso

| | Fabricar un asistente tipo GPT |
|---|---|
| **Tipo** | Modelo generativo de texto (decoder Transformer, Lección 12.4) |
| **Intuición** | Leer muchísimo → practicar con tutor → pulir con opiniones |
| **Cómo aprende** | 1) siguiente token sobre texto crudo; 2) siguiente token sobre ejemplos instrucción→respuesta; 3) maximizar un modelo de recompensa aprendido de preferencias |
| **Hiperparámetros clave** | `lr` (con *warmup*), tamaño de lote, longitud de contexto, tokens de entrenamiento, tamaño del modelo |
| **✅ Fortalezas** | No necesita etiquetas en la etapa 1 · Un solo modelo sirve para miles de tareas |
| **❌ Debilidades** | Costo enorme · Hereda sesgos y errores de los datos · *Reward hacking* · Alucina (Lección 13.9) |
| **Cuándo entrenar tú algo** | Casi nunca desde cero: parte de un modelo existente y ajústalo (LoRA, Lección 13.7) o usa RAG (13.8) |

## 9. 🧠 Resumen en 5 puntos
1. Un asistente se fabrica en **tres etapas**: pre-entrenamiento, ajuste por instrucciones (SFT) y RLHF/RLAIF.
2. El pre-entrenamiento solo **predice el siguiente token**, pero para hacerlo bien el modelo tiene que aprender gramática y conocimiento.
3. La **pérdida** (entropía cruzada) mide la sorpresa; la **perplejidad** = e^pérdida, "entre cuántas opciones duda". Al inicio vale ≈ ln(vocabulario).
4. El SFT enseña **el formato** de asistente con pocos ejemplos de calidad; el RLHF/RLAIF ajusta **el estilo** a partir de comparaciones.
5. Optimizar contra un modelo de recompensa imperfecto lleva a **reward hacking**: la métrica sube y la calidad no.

---
## 10. 🏋️ Ejercicios finales

### 10.1 Repaso rápido (opción múltiple)

**1.** ¿Qué datos usa el pre-entrenamiento?
- a) Pares pregunta–respuesta escritos por expertos · b) Texto crudo, sin etiquetas · c) Comparaciones "A mejor que B" · d) Imágenes etiquetadas

**2.** Un modelo con vocabulario de 1,000 tokens, sin entrenar, debería empezar con una pérdida cercana a:
- a) 0 · b) 1 · c) ln(1000) ≈ 6.9 · d) 1000

**3.** Tu pérdida de entrenamiento salta y llega a `nan` en los primeros pasos. ¿Qué cambias primero?
- a) Subir la tasa de aprendizaje · b) Bajar la tasa de aprendizaje · c) Agrandar el contexto · d) Quitar la validación

**4.** ¿Qué etapa convierte un modelo que "continúa textos" en uno que "sigue instrucciones"?
- a) Pre-entrenamiento · b) Tokenización · c) Ajuste por instrucciones (SFT) · d) Cuantización

**5.** En RLAIF, ¿quién da las preferencias?
- a) Usuarios pagados · b) Otro modelo de IA, guiado por principios · c) El propio texto de internet · d) Nadie: no hay preferencias

✍️ *Mis respuestas:* 1 → _ · 2 → _ · 3 → _ · 4 → _ · 5 → _

<details><summary><b>👀 Ver respuestas</b></summary>

1. **b)** Por eso puede usar billones de tokens: no hay que etiquetar nada.
2. **c)** Reparte la probabilidad por igual: −ln(1/1000) = ln(1000) ≈ 6.9.
3. **b)** Una pérdida que diverge es el síntoma clásico de un `lr` demasiado alto (lo viste con `lr=0.1`).
4. **c)** El SFT enseña el formato instrucción → respuesta.
5. **b)** *AI Feedback*: un modelo juez con una lista de principios ("constitución").
</details>

### 10.2 Preguntas para pensar (sin ejecutar)

**A.** Si el pre-entrenamiento solo enseña a "adivinar la siguiente palabra", ¿por qué el modelo termina sabiendo que la capital de Francia es París? Piensa en qué tendría que "saber" para predecir bien la palabra que sigue a *"La capital de Francia es"*.

**B. Caso de análisis 💼** Una empresa quiere un asistente para su servicio al cliente. Propone medir la calidad solo con la **satisfacción** que marcan los clientes (1–5 estrellas) y optimizar el modelo para subir ese número. ¿Qué podría salir mal? ¿Qué agregarías?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuestas</b></summary>

**A.** Para predecir bien millones de textos, la manera más "barata" de bajar la pérdida es **modelar el mundo que describen**: gramática, hechos, relaciones de causa. Saber que después de *"La capital de Francia es"* viene "París" baja la pérdida en miles de documentos. El conocimiento aparece como **efecto secundario** de comprimir el texto.

**B.** Es *reward hacking* en la vida real: el modelo puede aprender a **decir lo que el cliente quiere oír** (prometer reembolsos que no existen, adular, nunca decir "no") porque eso sube las estrellas. Conviene combinar varias señales: resolución real del problema, revisión humana de una muestra, reglas de negocio que no se pueden violar y una penalización por alejarse demasiado del modelo original.
</details>

### 10.3 🧩 Práctica integradora: tu propio mini-LLM

Vas a recorrer las tres etapas en miniatura. Cada paso tiene su respuesta: **no avances sin revisarla**.

**Paso 1 🔮 — ¿Cuánto entrenar?** Entrena el mini-modelo con 100, 500 y 3,000 pasos. ¿La pérdida de validación sigue bajando siempre igual de rápido?

In [ ]:
for pasos in [100, 500, 3000]:   # ✏️
    _, val = entrenar(pasos=pasos, mostrar=())
    print(f"{pasos:>5} pasos → pérdida de validación {val:.3f}")

<details><summary><b>👀 Ver respuesta</b></summary>

No. De 100 a 500 pasos la pérdida de validación baja mucho (≈0.85 → ≈0.31), pero de 500 a 3,000 casi no cambia (≈0.26). Son **rendimientos decrecientes**: con este modelo tan pequeño, más entrenamiento ya no ayuda; para mejorar habría que agrandarlo **y** darle más datos. Es la intuición de las leyes de escalamiento (Lección 12.5): tamaño del modelo, datos y cómputo deben crecer juntos.
</details>

**Paso 2 🔮 — Un texto raro para tu modelo.** Calcula la perplejidad del mini-modelo sobre una frase **del estilo del corpus** y sobre una **con palabras que nunca vio**. ¿Cuál será más alta?

In [ ]:
K = 8
def perplejidad_mini(modelo, texto):
    ids = torch.tensor([a_id.get(c, a_id[" "]) for c in texto])     # caracteres desconocidos → espacio
    X = torch.stack([ids[i:i + K] for i in range(len(ids) - K)]); Y = ids[K:]
    with torch.no_grad():
        return math.exp(nn.functional.cross_entropy(modelo(X), Y).item())

print("estilo del corpus:", round(perplejidad_mini(mini, "la abuela busca la luna en el campo."), 2))
print("palabras nuevas  :", round(perplejidad_mini(mini, "la abuela programa un robot en marte."), 2))

<details><summary><b>👀 Ver respuesta</b></summary>

La frase con palabras nuevas es muchísimo más sorprendente: perplejidad ≈**142** contra ≈**1.3** para la frase del estilo del corpus. El modelo nunca vio "programa", "robot" ni "marte" (y la "g" ni siquiera existe en su vocabulario: la cambiamos por un espacio).

Es lo mismo que le pasaba a GPT-2 con el español: **un modelo de lenguaje solo sabe de lo que había en sus datos de pre-entrenamiento.**
</details>

**Paso 3 ✋ — Tu propio "modelo de recompensa".** Modifica `BUENAS` y `MALAS` para premiar respuestas **cortas y concretas** (por ejemplo, resta un punto por cada 10 palabras) y vuelve a elegir la mejor candidata. ¿Cambia la ganadora?

In [ ]:
def recompensa_v2(texto):
    largo = len(texto.split())
    return recompensa(texto) - largo // 10        # ✏️ cambia la penalización por largo

for i, c in enumerate(candidatas_7):
    print(f"[{i}] v1 {recompensa(c):+d} · v2 {recompensa_v2(c):+d} │ {c[:70]}")

<details><summary><b>👀 Ver respuesta</b></summary>

Con `recompensa_v2` la [1] y la [2] pierden puntos por largas y bajan a 0, así que entre las respuestas reales gana la [0], *"I'll call you back."* (+1). Corta, sí, pero tampoco resuelve nada. Y la tramposa **sigue ganando** (+11): basta con que sea corta y esté llena de palabras clave.

Lección: arreglar un modelo de recompensa con más reglas simples suele abrir **otros** huecos. Por eso en la práctica se aprende de muchas comparaciones humanas y se revisa a mano lo que el modelo optimizado produce.
</details>

**Paso 4 ✋ — Ponlo en palabras.** Explica a un gerente, en 4 frases y sin tecnicismos, por qué *"el chatbot tiene muy buena calificación de nuestro modelo de recompensa"* **no** garantiza que sea un buen chatbot.

✍️ *Mi explicación:*

<details><summary><b>👀 Ver respuesta</b></summary>

Un ejemplo: *"El modelo de recompensa es un juez automático que aprendió a reconocer algunas señales de una buena respuesta, como disculparse u ofrecer ayuda. Si entrenamos el chatbot para complacer a ese juez, aprende a usar esas señales aunque no resuelva nada: en nuestro experimento, una respuesta que solo repetía 'sorry, refund, help' sacó la mejor nota. Por eso, además del juez automático, necesitamos revisar conversaciones reales y medir si el problema del cliente se resolvió. La calificación es una pista, no la meta."*
</details>

---
## 📚 Para profundizar
- 📺 **Video:** Andrej Karpathy, *"Let's build GPT: from scratch, in code, spelled out"* y *"Intro to Large Language Models"* (inglés, con subtítulos).
- 📄 **Artículo:** Ouyang et al. (2022), *Training language models to follow instructions with human feedback* (InstructGPT): las tres etapas por primera vez juntas.
- 📄 **Artículo:** Bai et al. (2022), *Constitutional AI*: el origen de RLAIF.
- 🧪 **Para Colab:** la librería `trl` (Hugging Face) implementa SFT, DPO y PPO para modelos reales.

---
⬅️ **Anterior:** [13.4 Modelos de difusión: crear imágenes quitando ruido](13.4-modelos-de-difusion.ipynb) · ➡️ **Siguiente:** [13.6 Hiperparámetros de generación: las perillas de la creatividad](13.6-hiperparametros-de-generacion.ipynb)